# Hörmander’s propagation of singularities: a microlocal tour

This notebook illustrates two fundamental theorems of microlocal analysis using the `microlocal` and `psiop` packages:

1. **The characteristic variety** $\operatorname{Char}(P) = \{(x,\xi): p(x,\xi)=0\}$ is where singularities can live.
2. **The wave‑front set** $\operatorname{WF}(u)$ propagates along the bicharacteristic flow of the principal symbol $p$.

We will work with a concrete **one‑dimensional wave operator** (fixed frequency) and follow an initial point singularity through phase space.

In [ ]:
# Core imports
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

# Microlocal and pseudo‑differential toolkits
from microlocal import (
    characteristic_variety, bicharacteristic_flow,
    plot_characteristic_set, plot_wavefront_set, propagate_singularity,
    compute_maslov_index, compute_caustics_2d
)
from psiop import PseudoDifferentialOperator

# Nice plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. The operator and its principal symbol

Consider the **1‑D wave operator** with a fixed time frequency $\omega = 1$:
$$
P = \partial_t^2 - \partial_x^2 \quad\longrightarrow\quad \text{after Fourier in }t:\; p(x,\xi) = \xi^2 - 1.
$$
The principal symbol is $p(x,\xi) = \xi^2 - 1$ (it does not depend on $x$).

In [ ]:
x, xi = sp.symbols('x xi', real=True)
p = xi**2 - 1
print("Principal symbol p(x,ξ) =", p)

## 2. Characteristic variety

$\operatorname{Char}(P) = \{(x,\xi): \xi^2-1=0\} = \{ \xi = 1 \} \cup \{ \xi = -1 \}$.

The `characteristic_variety` function gives us the equation and explicit solutions.

In [ ]:
char = characteristic_variety(p, dim=1)
print("Equation:", char['equation'])
print("Explicit ξ(x):", char['explicit'])

In [ ]:
# Visualise the characteristic set (red contour) and the symbol magnitude
plot_characteristic_set(p, x_range=(-2,2), xi_range=(-2,2), resolution=200)

The red lines ($\xi = \pm 1$) are the only phase‑space points where the principal symbol vanishes. According to Hörmander’s theorem, the wave‑front set of any solution of $P u = 0$ must lie on these lines.

## 3. Bicharacteristic flow

The Hamiltonian vector field is
$$
\dot{x} = \frac{\partial p}{\partial \xi} = 2\xi,\qquad 
\dot{\xi} = -\frac{\partial p}{\partial x} = 0.
$$
Thus $\xi$ is constant, and $x(t) = x_0 + 2\xi_0\,t$.

Let us integrate a few bicharacteristics starting from different points on the characteristic variety.

In [ ]:
# Initial points: (x0, ξ0) with ξ0 = ±1
init_points = [(-1.5, 1.0), (-0.5, 1.0), (0.5, -1.0), (1.5, -1.0)]
tspan = (0, 2.0)

fig, ax = plt.subplots(figsize=(8,6))
colors = plt.cm.viridis(np.linspace(0,1,len(init_points)))

for (x0, xi0), col in zip(init_points, colors):
    traj = bicharacteristic_flow(p, (x0, xi0), tspan, dim=1, method='symplectic', n_steps=200)
    ax.plot(traj['x'], traj['xi'], color=col, lw=2, alpha=0.7)
    ax.plot(traj['x'][0], traj['xi'][0], 'o', color=col, ms=8)
    ax.plot(traj['x'][-1], traj['xi'][-1], 's', color=col, ms=6)

ax.set_xlabel('x'); ax.set_ylabel('ξ')
ax.set_title('Bicharacteristics of the wave operator (ξ²-1)')
ax.grid(alpha=0.3)
plt.show()

The trajectories are horizontal lines ($\xi = \pm 1$ constant) as expected. The endpoints (squares) have moved to the right ($\xi=+1$) or left ($\xi=-1$) depending on the sign of $\xi$.

## 4. Propagation of the wave‑front set

We now choose an **initial wave‑front set** consisting of a single point $(x_0,\xi_0) = (0, 1)$. The `propagate_singularity` function follows this point along the bicharacteristic flow, and `plot_wavefront_set` shows the whole trajectory.

In [ ]:
initial_wf = [(0.0, 1.0)]
tspan = (0, 3.0)

# Propagate and visualise in phase space
plot_wavefront_set(p, initial_wf, tspan, dim=1, projection='cotangent',
                   show_flow=True, show_endpoints=True,
                   title='Wave-front set evolution: (0,1) → (6,1)')

The green dot is the initial singularity, the red square is its location at $t=3$ (at $x=6$, $\xi=1$). The entire trajectory (blue line) lies on the characteristic set $\xi=1$ – **the wave‑front set propagates along the bicharacteristic**.

## 5. Using the `PseudoDifferentialOperator` class

The `psiop` module allows us to build a full pseudo‑differential operator from the symbol and apply it to a function. This demonstrates how the operator *smoothes* singularities not lying on the characteristic set.

In [ ]:
# Create the operator
op = PseudoDifferentialOperator(expr=p, vars_x=[x], mode='symbol')
print("Symbol:", op.symbol)
print("Principal symbol order:", op.symbol_order())

In [ ]:
# Define a function with a singularity at x=0 (e.g., a step)
N = 256
L = 10.0
x_vals = np.linspace(-L, L, N, endpoint=False)
u = np.heaviside(x_vals, 0.5)   # step function (singular at 0)

# Frequency grid for FFT
dx = x_vals[1] - x_vals[0]
k = np.fft.fftfreq(N, d=dx) * 2*np.pi

# Apply the operator (periodic BC, fast path because symbol is constant in x)
Pu = op.apply(u, x_vals, k, boundary_condition='periodic')

# Plot
plt.figure(figsize=(10,4))
plt.plot(x_vals, u.real, label='u(x) = H(x)')
plt.plot(x_vals, Pu.real, label='P u(x)', linestyle='--')
plt.xlabel('x'); plt.ylabel('amplitude')
plt.title(r'Action of $P = \xi^2-1$ on a step function')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 6. Schrödinger operator: harmonic oscillator

Consider the symbol of the quantum harmonic oscillator at energy $E=1$:
$$
p(x,\xi) = \xi^2 + x^2 - 1.
$$
The characteristic variety is the circle $x^2+\xi^2 = 1$. The bicharacteristic flow is
$$
\dot{x} = 2\xi,\qquad \dot{\xi} = -2x,
$$
which yields circular motion with period $\pi$.

In [ ]:
p_harm = xi**2 + x**2 - 1
plot_characteristic_set(p_harm, x_range=(-1.5,1.5), xi_range=(-1.5,1.5), resolution=200)

In [ ]:
# Initial point on the circle
init_harm = [(1.0, 0.0)]
tspan_harm = (0, 3.0)

plot_wavefront_set(p_harm, init_harm, tspan_harm, dim=1, projection='cotangent',
                   show_flow=True, show_endpoints=True,
                   title='Harmonic oscillator: circular propagation')

The singularity travels around the circle, returning to its starting point after $t=\pi$ (the period of the classical motion).

## 7. 2D wave operator with circular wave‑front set

Now move to two dimensions. The symbol of the wave operator (fixed frequency) is
$$
p(x,y,\xi,\eta) = \xi^2 + \eta^2 - 1.
$$
We choose an initial wave‑front set that is a **circle in position space** with matching frequency direction (outgoing radial wave):
$$
(x_0, y_0) = (\cos\theta, \sin\theta),\quad (\xi_0, \eta_0) = (\cos\theta, \sin\theta).
$$
This corresponds to a circular front expanding outward.

In [ ]:
# Define 2D symbol
x, y, xi, eta = sp.symbols('x y xi eta', real=True)
p2 = xi**2 + eta**2 - 1

# Generate initial points on a circle
theta = np.linspace(0, 2*np.pi, 24, endpoint=False)
init_circle = [(np.cos(t), np.sin(t), np.cos(t), np.sin(t)) for t in theta]
tspan2 = (0, 2.0)

# Plot the propagation in position space
plot_wavefront_set(p2, init_circle, tspan2, dim=2, projection='position',
                   show_flow=True, show_endpoints=True,
                   title='2D wave operator: expanding circular front')

The green dots mark the initial circle, the red squares the position after $t=2$ – the circle has expanded (radius increased from 1 to 3), confirming that singularities travel outward with speed 1 along the bicharacteristics.

## 8. Animated Wavefront Propagation for a Non‑Standard 2D Hamiltonian

This animation visualizes the evolution of an initially circular wave‑front set under a modified 2D wave operator with symbol  
$ p(x,y,\xi,\eta) = \xi^2 + \eta^2 + x\eta - y\xi $.  
The additional linear terms couple position and momentum, causing the wavefront to deform and rotate as it expands. Each ray is colored by its initial angle, and the current wavefront is marked by a dashed white circle. The green dots indicate the starting points (unit circle), while the red squares trace the instantaneous position of each ray. The animation clearly shows how the singularity distribution evolves under the bicharacteristic flow of this non‑separable Hamiltonian.

In [ ]:

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from microlocal import bicharacteristic_flow

# ----- Setup: same as in plot_wavefront_set -----
plt.style.use('dark_background')
x, y, xi, eta = sp.symbols('x y xi eta', real=True)
p2 = xi**2 + eta**2 + x*eta-y*xi

n_rays = 72
theta = np.linspace(0, 2*np.pi, n_rays, endpoint=False)
init_points = [(np.cos(t), np.sin(t), np.cos(t), np.sin(t)) for t in theta]
tspan = (0, 2.0)
n_steps = 300

# Precompute trajectories
trajs = []
for z0 in init_points:
    traj = bicharacteristic_flow(p2, z0, tspan, dim=2, method='symplectic', n_steps=n_steps)
    trajs.append(traj)

t_grid = trajs[0]['t']
colors = plt.cm.viridis(np.linspace(0, 1, n_rays))  # or 'plasma' – mimic plot_wavefront_set

# Create figure and axes
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('2D wave operator: expanding circular front (animation)', fontsize=14)
ax.grid(alpha=0.3)

# Initialize line objects (one per ray)
lines = []
for i in range(n_rays):
    line, = ax.plot([], [], color=colors[i], alpha=0.7, lw=1.2)
    lines.append(line)

# Start and end point markers (green circles, red squares)
start_scat = ax.scatter([], [], s=20, c='limegreen', marker='o', alpha=0.6, zorder=5)
end_scat = ax.scatter([], [], s=25, c='crimson', marker='s', alpha=0.8, zorder=5)

# Dashed wavefront circle
wavefront, = ax.plot([], [], '--', color='white', linewidth=1, alpha=0.5)

# Update function
def update(frame):
    # Update each ray's trajectory up to current frame
    for i, traj in enumerate(trajs):
        lines[i].set_data(traj['x'][:frame+1], traj['y'][:frame+1])
    
    # Update start points (all fixed at t=0)
    start_scat.set_offsets(np.c_[[t['x'][0] for t in trajs], [t['y'][0] for t in trajs]])
    
    # Update end points (current positions)
    end_scat.set_offsets(np.c_[[t['x'][frame] for t in trajs], [t['y'][frame] for t in trajs]])
    
    # Update wavefront circle: radius = distance of any ray at this time
    r = np.hypot(trajs[0]['x'][frame], trajs[0]['y'][frame])
    circ_theta = np.linspace(0, 2*np.pi, 100)
    wavefront.set_data(r * np.cos(circ_theta), r * np.sin(circ_theta))
    
    ax.set_title(f'2D wave operator: expanding circular front\nt = {t_grid[frame]:.2f}', fontsize=14)
    return lines + [start_scat, end_scat, wavefront]

# Create animation
anim = FuncAnimation(fig, update, frames=n_steps, interval=20, blit=False, repeat=True)

# Display in Jupyter
from IPython.display import HTML
HTML(anim.to_html5_video())

## 9. Conclusion

We have illustrated the core ideas of Hörmander’s theory:

- The **characteristic variety** is the zero set of the principal symbol.
- The **bicharacteristic flow** moves along $\dot{x} = \partial_\xi p,\; \dot{\xi} = -\partial_x p$.
- The **wave‑front set** of a solution to $Pu=0$ is a union of whole bicharacteristics.
- The `microlocal` and `psiop` packages provide a seamless symbolic‑numerical environment to explore these concepts in 1D and 2D.

For further experiments, try:
- A Schrödinger operator with a non‑quadratic potential (e.g., $V(x)=x^4$).
- A 2D operator with variable coefficients, e.g., $p = \xi^2+\eta^2 - c(x,y)$.
- Compute the full caustic set as a curve in position space.